In [ ]:
# ============================================================
# BRANCH A — DIRECT DOCUMENT REPRESENTATION
# D12 — Our World in Data Annual CO2 Emissions Dataset
# ============================================================
#
# Methodology stages covered:
# Stage 2 — Branch A: Direct Ingestion
# Stage 3 — Information Extraction using an LLM
# Post-extraction technical diagnostics
# ============================================================

from google.colab import files
from pathlib import Path
from collections import Counter

import csv
import hashlib
import json
import platform
import sys

import pandas as pd


In [ ]:
# ============================================================
# 1. Configuration
# ============================================================

DOCUMENT_ID = "D12"

DOCUMENT_NAME = (
    "Our World in Data — Annual CO2 emissions time series"
)

BRANCH = "A"

BRANCH_NAME = "Direct Ingestion"

SOURCE_FORMAT = ".csv"

INPUT_REPRESENTATION = "Original CSV file"

DIRECT_DOCUMENT_INGESTION = True


# ------------------------------------------------------------
# Stage 1 source expectations
# ------------------------------------------------------------

EXPECTED_SOURCE_ROW_COUNT = 275

EXPECTED_SOURCE_COLUMNS = [
    "Year",
    "Annual CO2 emissions"
]

EXPECTED_MIN_YEAR = 1750

EXPECTED_MAX_YEAR = 2024


# ------------------------------------------------------------
# Fixed Stage 1 extraction scope
# ------------------------------------------------------------

TARGET_YEARS = [
    1750,
    1800,
    1850,
    1900,
    1950,
    1960,
    1970,
    1980,
    1990,
    2000,
    2010,
    2011,
    2012,
    2013,
    2014,
    2015,
    2016,
    2017,
    2018,
    2019,
    2020,
    2021,
    2022,
    2023,
    2024
]


EXPECTED_RECORD_COUNT = len(
    TARGET_YEARS
)


# ------------------------------------------------------------
# Exact Stage 1 category
# ------------------------------------------------------------

REFERENCE_CATEGORY = (
    "Environmental time-series"
)


EXPECTED_CATEGORY_COUNTS = {
    REFERENCE_CATEGORY:
        EXPECTED_RECORD_COUNT
}


REFERENCE_TOPIC = (
    "Annual CO2 emissions"
)


REFERENCE_DESCRIPTION = (
    "Annual CO2 emissions"
)


REFERENCE_UNIT = None


# ------------------------------------------------------------
# Fixed Stage 1 schema
# ------------------------------------------------------------

EXPECTED_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Value",
    "Unit",
    "Reporting Period",
    "Source Location"
]


STRING_OR_NULL_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Unit",
    "Reporting Period",
    "Source Location"
]


MANDATORY_CONTENT_FIELDS = [
    "Category",
    "Topic",
    "Description",
    "Reporting Period",
    "Source Location"
]


ALLOWED_CATEGORIES = {
    REFERENCE_CATEGORY
}


# ------------------------------------------------------------
# Output paths
# ------------------------------------------------------------

OUTPUT_DIR = Path(
    "outputs_D12_branch_A"
)

OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


INPUT_INTEGRITY_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_input_integrity.json"
)

REPRESENTATION_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_representation.json"
)

DATASET_PROFILE_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_dataset_profile.csv"
)

PROMPT_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_prompt.txt"
)

RAW_RESPONSE_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_raw_response.txt"
)

PARSED_EXTRACTION_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_parsed_extraction.json"
)

TECHNICAL_DIAGNOSTICS_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_technical_diagnostics.json"
)

YEAR_CHECK_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_year_check.csv"
)

EXPERIMENT_METADATA_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_experiment_metadata.json"
)

EXPERIMENT_SUMMARY_PATH = (
    OUTPUT_DIR
    / "D12_branch_A_experiment_summary.json"
)


print("Document:", DOCUMENT_ID)
print("Branch:", BRANCH)
print("Representation:", INPUT_REPRESENTATION)
print("Expected source rows:", EXPECTED_SOURCE_ROW_COUNT)
print("Target-year count:", EXPECTED_RECORD_COUNT)
print("Expected fields:", len(EXPECTED_FIELDS))

In [ ]:
# ============================================================
# 2. Source document and integrity diagnostics
# ============================================================

print(
    "Upload the original D12 CSV file."
)


uploaded = files.upload()


csv_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".csv")
]


if len(csv_paths) != 1:

    raise ValueError(
        "Upload exactly one CSV source document."
    )


SOURCE_PATH = csv_paths[0]


# ------------------------------------------------------------
# SHA-256 utility
# ------------------------------------------------------------

def sha256_file(path):

    digest = hashlib.sha256()

    with path.open("rb") as file:

        for chunk in iter(
            lambda: file.read(
                1024 * 1024
            ),
            b""
        ):

            digest.update(
                chunk
            )

    return digest.hexdigest()


SOURCE_SHA256 = sha256_file(
    SOURCE_PATH
)


FILE_SIZE_BYTES = (
    SOURCE_PATH.stat().st_size
)


FILE_NON_EMPTY = (
    FILE_SIZE_BYTES > 0
)


# ------------------------------------------------------------
# Detect delimiter diagnostically
# ------------------------------------------------------------

raw_sample = (
    SOURCE_PATH.read_text(
        encoding="utf-8",
        errors="replace"
    )[:5000]
)


try:

    dialect = csv.Sniffer().sniff(
        raw_sample,
        delimiters=[
            ";",
            ",",
            "\t",
            "|"
        ]
    )

    DETECTED_DELIMITER = (
        dialect.delimiter
    )


except csv.Error:

    DETECTED_DELIMITER = ";"


# ------------------------------------------------------------
# Diagnostic dataframe loading
# ------------------------------------------------------------

df = pd.read_csv(
    SOURCE_PATH,
    sep=DETECTED_DELIMITER
)


OBSERVED_SOURCE_ROW_COUNT = len(
    df
)


OBSERVED_SOURCE_COLUMNS = list(
    df.columns
)


SOURCE_ROW_COUNT_VALID = (
    OBSERVED_SOURCE_ROW_COUNT
    == EXPECTED_SOURCE_ROW_COUNT
)


SOURCE_COLUMNS_VALID = (
    OBSERVED_SOURCE_COLUMNS
    == EXPECTED_SOURCE_COLUMNS
)


# ------------------------------------------------------------
# Diagnostic numeric parsing only
# ------------------------------------------------------------

diagnostic_df = df.copy()


diagnostic_df[
    "Year"
] = pd.to_numeric(
    diagnostic_df[
        "Year"
    ],
    errors="coerce"
)


diagnostic_df[
    "Annual CO2 emissions"
] = pd.to_numeric(
    diagnostic_df[
        "Annual CO2 emissions"
    ],
    errors="coerce"
)


# ------------------------------------------------------------
# Source integrity
# ------------------------------------------------------------

NULL_CELL_COUNT = int(
    diagnostic_df.isna().sum().sum()
)


NULL_VALUES_ABSENT = (
    NULL_CELL_COUNT == 0
)


DUPLICATE_ROW_COUNT = int(
    diagnostic_df.duplicated().sum()
)


DUPLICATE_ROWS_ABSENT = (
    DUPLICATE_ROW_COUNT == 0
)


DUPLICATE_YEAR_COUNT = int(
    diagnostic_df[
        "Year"
    ].duplicated().sum()
)


DUPLICATE_YEARS_ABSENT = (
    DUPLICATE_YEAR_COUNT == 0
)


observed_years = (
    diagnostic_df[
        "Year"
    ]
    .dropna()
    .astype(int)
    .tolist()
)


OBSERVED_MIN_YEAR = (
    min(observed_years)
    if observed_years
    else None
)


OBSERVED_MAX_YEAR = (
    max(observed_years)
    if observed_years
    else None
)


YEAR_RANGE_VALID = all([
    OBSERVED_MIN_YEAR
    == EXPECTED_MIN_YEAR,

    OBSERVED_MAX_YEAR
    == EXPECTED_MAX_YEAR
])


EXPECTED_FULL_YEAR_SEQUENCE = list(
    range(
        EXPECTED_MIN_YEAR,
        EXPECTED_MAX_YEAR + 1
    )
)


MISSING_SOURCE_YEARS = sorted(
    set(
        EXPECTED_FULL_YEAR_SEQUENCE
    )
    - set(
        observed_years
    )
)


UNEXPECTED_SOURCE_YEARS = sorted(
    set(
        observed_years
    )
    - set(
        EXPECTED_FULL_YEAR_SEQUENCE
    )
)


YEAR_SEQUENCE_COMPLETE = (
    len(
        MISSING_SOURCE_YEARS
    )
    == 0
    and len(
        UNEXPECTED_SOURCE_YEARS
    )
    == 0
)


CHRONOLOGICAL_ORDER_VALID = (
    observed_years
    == sorted(
        observed_years
    )
)


TARGET_YEARS_PRESENT_IN_SOURCE = (
    set(
        TARGET_YEARS
    ).issubset(
        set(
            observed_years
        )
    )
)


# ------------------------------------------------------------
# Build notebook-side source-row lookup
# ------------------------------------------------------------

SOURCE_ROW_BY_YEAR = {}


for dataframe_index, row in (
    diagnostic_df.iterrows()
):

    if pd.isna(
        row["Year"]
    ):
        continue


    year = int(
        row["Year"]
    )


    SOURCE_ROW_BY_YEAR[
        year
    ] = (
        int(
            dataframe_index
        )
        + 2
    )


EXPECTED_SOURCE_LOCATION_BY_YEAR = {
    year:
        (
            f"CSV data row "
            f"{SOURCE_ROW_BY_YEAR[year]}"
        )

    for year
    in TARGET_YEARS
    if year
    in SOURCE_ROW_BY_YEAR
}


# ------------------------------------------------------------
# Diagnostic source-value lookup
# ------------------------------------------------------------

SOURCE_VALUE_BY_YEAR = {}


for _, row in (
    diagnostic_df.iterrows()
):

    if (
        pd.isna(
            row["Year"]
        )
        or pd.isna(
            row[
                "Annual CO2 emissions"
            ]
        )
    ):
        continue


    SOURCE_VALUE_BY_YEAR[
        int(
            row["Year"]
        )
    ] = (
        float(
            row[
                "Annual CO2 emissions"
            ]
        )
    )


# ------------------------------------------------------------
# Dataset profile
# ------------------------------------------------------------

DATASET_PROFILE = pd.DataFrame(
    [
        {
            "Metric":
                "File size bytes",

            "Value":
                FILE_SIZE_BYTES
        },
        {
            "Metric":
                "Detected delimiter",

            "Value":
                DETECTED_DELIMITER
        },
        {
            "Metric":
                "Observed rows",

            "Value":
                OBSERVED_SOURCE_ROW_COUNT
        },
        {
            "Metric":
                "Observed columns",

            "Value":
                len(
                    OBSERVED_SOURCE_COLUMNS
                )
        },
        {
            "Metric":
                "Minimum year",

            "Value":
                OBSERVED_MIN_YEAR
        },
        {
            "Metric":
                "Maximum year",

            "Value":
                OBSERVED_MAX_YEAR
        },
        {
            "Metric":
                "Null cells",

            "Value":
                NULL_CELL_COUNT
        },
        {
            "Metric":
                "Duplicate rows",

            "Value":
                DUPLICATE_ROW_COUNT
        },
        {
            "Metric":
                "Duplicate years",

            "Value":
                DUPLICATE_YEAR_COUNT
        },
        {
            "Metric":
                "Missing years",

            "Value":
                len(
                    MISSING_SOURCE_YEARS
                )
        }
    ]
)


DATASET_PROFILE.to_csv(
    DATASET_PROFILE_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# Input-integrity result
# ------------------------------------------------------------

INPUT_INTEGRITY_PASSED = all([
    FILE_NON_EMPTY,
    SOURCE_ROW_COUNT_VALID,
    SOURCE_COLUMNS_VALID,
    NULL_VALUES_ABSENT,
    DUPLICATE_ROWS_ABSENT,
    DUPLICATE_YEARS_ABSENT,
    YEAR_RANGE_VALID,
    YEAR_SEQUENCE_COMPLETE,
    CHRONOLOGICAL_ORDER_VALID,
    TARGET_YEARS_PRESENT_IN_SOURCE
])


INPUT_INTEGRITY = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_file":
        SOURCE_PATH.name,

    "input_file_sha256":
        SOURCE_SHA256,

    "input_representation":
        INPUT_REPRESENTATION,

    "source_format":
        SOURCE_FORMAT,

    "file_size_bytes":
        FILE_SIZE_BYTES,

    "file_non_empty":
        FILE_NON_EMPTY,

    "detected_delimiter":
        DETECTED_DELIMITER,

    "expected_source_row_count":
        EXPECTED_SOURCE_ROW_COUNT,

    "observed_source_row_count":
        OBSERVED_SOURCE_ROW_COUNT,

    "source_row_count_valid":
        SOURCE_ROW_COUNT_VALID,

    "expected_source_columns":
        EXPECTED_SOURCE_COLUMNS,

    "observed_source_columns":
        OBSERVED_SOURCE_COLUMNS,

    "source_columns_valid":
        SOURCE_COLUMNS_VALID,

    "null_cell_count":
        NULL_CELL_COUNT,

    "null_values_absent":
        NULL_VALUES_ABSENT,

    "duplicate_row_count":
        DUPLICATE_ROW_COUNT,

    "duplicate_rows_absent":
        DUPLICATE_ROWS_ABSENT,

    "duplicate_year_count":
        DUPLICATE_YEAR_COUNT,

    "duplicate_years_absent":
        DUPLICATE_YEARS_ABSENT,

    "expected_min_year":
        EXPECTED_MIN_YEAR,

    "expected_max_year":
        EXPECTED_MAX_YEAR,

    "observed_min_year":
        OBSERVED_MIN_YEAR,

    "observed_max_year":
        OBSERVED_MAX_YEAR,

    "year_range_valid":
        YEAR_RANGE_VALID,

    "missing_source_years":
        MISSING_SOURCE_YEARS,

    "unexpected_source_years":
        UNEXPECTED_SOURCE_YEARS,

    "year_sequence_complete":
        YEAR_SEQUENCE_COMPLETE,

    "chronological_order_valid":
        CHRONOLOGICAL_ORDER_VALID,

    "target_years_present_in_source":
        TARGET_YEARS_PRESENT_IN_SOURCE,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED
}


INPUT_INTEGRITY_PATH.write_text(
    json.dumps(
        INPUT_INTEGRITY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    "Source:",
    SOURCE_PATH.name
)

print(
    "SHA-256:",
    SOURCE_SHA256
)

print(
    "Detected delimiter:",
    repr(
        DETECTED_DELIMITER
    )
)

print(
    "Observed rows:",
    OBSERVED_SOURCE_ROW_COUNT
)

print(
    "Observed columns:",
    OBSERVED_SOURCE_COLUMNS
)

print(
    "Year range:",
    OBSERVED_MIN_YEAR,
    "to",
    OBSERVED_MAX_YEAR
)

print(
    "Missing years:",
    len(
        MISSING_SOURCE_YEARS
    )
)

print(
    "Duplicate years:",
    DUPLICATE_YEAR_COUNT
)

print(
    "Null cells:",
    NULL_CELL_COUNT
)

print(
    "Input integrity passed:",
    INPUT_INTEGRITY_PASSED
)


display(
    DATASET_PROFILE
)


if not FILE_NON_EMPTY:

    raise AssertionError(
        "D12 source CSV is empty."
    )


if not SOURCE_ROW_COUNT_VALID:

    raise AssertionError(
        f"Expected "
        f"{EXPECTED_SOURCE_ROW_COUNT} "
        f"source rows, found "
        f"{OBSERVED_SOURCE_ROW_COUNT}."
    )


if not SOURCE_COLUMNS_VALID:

    raise AssertionError(
        "Unexpected D12 source columns."
    )


if not YEAR_SEQUENCE_COMPLETE:

    raise AssertionError(
        "D12 source year sequence is incomplete "
        "or contains unexpected years."
    )


if not TARGET_YEARS_PRESENT_IN_SOURCE:

    raise AssertionError(
        "One or more predefined target years "
        "are missing from the D12 source."
    )

In [ ]:
# ============================================================
# 3. Branch A representation
# ============================================================

REPRESENTATION = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "representation_type":
        "Original source document",

    "input_file":
        SOURCE_PATH.name,

    "input_format":
        SOURCE_FORMAT,

    "source_representation":
        (
            "Semicolon-delimited annual time-series CSV "
            "with Year and Annual CO2 emissions columns"
        ),

    "complete_original_document_supplied":
        True,

    "direct_document_ingestion":
        True,

    "diagnostic_csv_parsing_applied":
        True,

    "diagnostic_delimiter_detection_applied":
        True,

    "diagnostic_numeric_parsing_applied":
        True,

    "diagnostically_parsed_dataframe_used_as_model_input":
        False,

    "derived_representation_used_as_model_input":
        False,

    "csv_to_text_conversion_applied":
        False,

    "row_filtering_applied_to_model_input":
        False,

    "column_filtering_applied_to_model_input":
        False,

    "column_restructuring_applied":
        False,

    "row_reordering_applied":
        False,

    "aggregation_applied":
        False,

    "interpolation_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_inference_applied":
        False,

    "value_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "model_input_description": (
        "The complete original D12 CSV is supplied directly to "
        "the LLM. Pandas loading, delimiter detection, datatype "
        "inspection and numeric parsing are used only for "
        "source-integrity diagnostics inside the notebook. "
        "The diagnostically parsed dataframe is not supplied "
        "to the model. No rows or columns are filtered from "
        "the source file before model ingestion."
    )
}


REPRESENTATION_PATH.write_text(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


print(
    json.dumps(
        REPRESENTATION,
        ensure_ascii=False,
        indent=2
    )
)

In [ ]:
# ============================================================
# 4. Extraction prompt
# ============================================================

TARGET_YEAR_TEXT = "\n".join(
    f"- {year}"
    for year in TARGET_YEARS
)


BRANCH_A_PROMPT = f"""You are an information extraction assistant.

Extract the predefined annual CO2-emissions observations represented
in the attached original CSV file.

Treat the attached original CSV as the only source of information.

The source contains an annual time series with the columns:

- Year
- Annual CO2 emissions

Extract one record for every year in the predefined target-year list
below.

Target years:

{TARGET_YEAR_TEXT}


For every included record return exactly these fields:

- Category
- Topic
- Description
- Value
- Unit
- Reporting Period
- Source Location


Use these fixed semantic-field values for every extracted record:

Category:
Environmental time-series

Topic:
Annual CO2 emissions

Description:
Annual CO2 emissions

Unit:
null


Field rules:

Category:
- Use exactly:
  "Environmental time-series"

Topic:
- Use exactly:
  "Annual CO2 emissions"

Description:
- Use exactly:
  "Annual CO2 emissions"

Value:
- Extract the value represented in the "Annual CO2 emissions"
  column for the corresponding target year.
- Preserve the source numerical value.
- Return the value as a JSON number.
- Do not calculate, interpolate, estimate, rescale, round or convert
  the source value.
- Do not use values from neighbouring years.

Unit:
- Return null.
- The supplied CSV does not explicitly represent a separate
  measurement-unit field.
- Do not infer or introduce a unit from external knowledge.

Reporting Period:
- Use the corresponding target year as a string.
- Example format:
  "1750"

Source Location:
- Identify the physical data row in the original CSV.
- The column-header row is physical CSV row 1.
- The first data row is physical CSV row 2.
- Use exactly this format:
  "CSV data row N"
- Determine N from the original source file.
- Do not infer row numbers from the target-year-list position.


Extraction rules:

- Extract only the predefined target years.
- Return one record for every year in the target-year list.
- Do not omit a listed year.
- Do not return years outside the predefined list.
- Use only values explicitly represented in the original CSV.
- Preserve the direct year-to-value association.
- Do not calculate missing values.
- Do not interpolate between years.
- Do not aggregate years.
- Do not introduce measurement units that are absent from the CSV.
- Do not use external knowledge.
- Do not follow external links.
- Do not modify or normalise source numerical values.
- Verify that every listed target year has been processed.
- Return only valid JSON.
- Do not include Markdown fences, explanations or commentary.
- Keep the exact field names and field order defined below.

Expected JSON structure:

{{
  "document_id": "D12",
  "branch": "A",
  "records": [
    {{
      "Category": "Environmental time-series",
      "Topic": "Annual CO2 emissions",
      "Description": "Annual CO2 emissions",
      "Value": null,
      "Unit": null,
      "Reporting Period": null,
      "Source Location": null
    }}
  ]
}}

Return only the JSON object.
"""


PROMPT_PATH.write_text(
    BRANCH_A_PROMPT,
    encoding="utf-8"
)


PROMPT_SHA256 = sha256_file(
    PROMPT_PATH
)


print(
    "Prompt saved:",
    PROMPT_PATH.name
)

print(
    "Prompt SHA-256:",
    PROMPT_SHA256
)

print()

print(
    BRANCH_A_PROMPT
)

## Independent Branch A extraction

Open a new independent conversation.

Upload:

1. the complete original D12 CSV;
2. `D12_branch_A_prompt.txt`.

Submit the prompt once.

Save complete untouched model response as:

`D12_branch_A_raw_response.txt`


In [ ]:
# ============================================================
# 5. Raw response preservation and parsing
# ============================================================

print(
    "Upload the untouched "
    "D12_branch_A_raw_response.txt file."
)


uploaded = files.upload()


txt_paths = [
    Path(name)
    for name in uploaded
    if name.lower().endswith(".txt")
]


if len(txt_paths) != 1:

    raise ValueError(
        "Upload exactly one TXT raw-response file."
    )


UPLOADED_RAW_RESPONSE_PATH = (
    txt_paths[0]
)


raw_response_text = (
    UPLOADED_RAW_RESPONSE_PATH.read_text(
        encoding="utf-8"
    )
)


if not raw_response_text.strip():

    raise ValueError(
        "The uploaded raw response is empty."
    )


# ------------------------------------------------------------
# Preserve untouched response BEFORE parsing
# ------------------------------------------------------------

RAW_RESPONSE_PATH.write_text(
    raw_response_text,
    encoding="utf-8"
)


RAW_RESPONSE_SHA256 = sha256_file(
    RAW_RESPONSE_PATH
)


# ------------------------------------------------------------
# Non-crashing JSON parse
# ------------------------------------------------------------

valid_json = False

json_parsing_error = None

parsed_response = None


try:

    parsed_response = json.loads(
        raw_response_text
    )

    valid_json = True


except json.JSONDecodeError as error:

    json_parsing_error = str(
        error
    )


# ------------------------------------------------------------
# Standard wrapper
# ------------------------------------------------------------

top_level_object_valid = (
    valid_json
    and isinstance(
        parsed_response,
        dict
    )
)


document_id_present = (
    top_level_object_valid
    and "document_id"
    in parsed_response
)


document_id_correct = (
    top_level_object_valid
    and parsed_response.get(
        "document_id"
    )
    == DOCUMENT_ID
)


branch_present = (
    top_level_object_valid
    and "branch"
    in parsed_response
)


branch_correct = (
    top_level_object_valid
    and parsed_response.get(
        "branch"
    )
    == BRANCH
)


records_present = (
    top_level_object_valid
    and "records"
    in parsed_response
)


records_is_list = (
    top_level_object_valid
    and isinstance(
        parsed_response.get(
            "records"
        ),
        list
    )
)


records_evaluable = (
    valid_json
    and top_level_object_valid
    and records_present
    and records_is_list
)


if records_evaluable:

    extracted_records = (
        parsed_response[
            "records"
        ]
    )

    observed_record_count = len(
        extracted_records
    )


else:

    extracted_records = []

    observed_record_count = None


# ------------------------------------------------------------
# Parsed extraction only when evaluable
# ------------------------------------------------------------

parsed_extraction_created = False

parsed_extraction_sha256 = None


if records_evaluable:

    PARSED_EXTRACTION = {
        "document_id":
            parsed_response.get(
                "document_id"
            ),

        "branch":
            parsed_response.get(
                "branch"
            ),

        "records":
            extracted_records
    }


    PARSED_EXTRACTION_PATH.write_text(
        json.dumps(
            PARSED_EXTRACTION,
            ensure_ascii=False,
            indent=2
        ),
        encoding="utf-8"
    )


    parsed_extraction_created = True


    parsed_extraction_sha256 = (
        sha256_file(
            PARSED_EXTRACTION_PATH
        )
    )


print(
    "Raw response SHA-256:",
    RAW_RESPONSE_SHA256
)

print(
    "Valid JSON:",
    valid_json
)

print(
    "JSON parsing error:",
    json_parsing_error
)

print(
    "Top-level object valid:",
    top_level_object_valid
)

print(
    "Document ID correct:",
    document_id_correct
)

print(
    "Branch correct:",
    branch_correct
)

print(
    "Records evaluable:",
    records_evaluable
)

print(
    "Observed record count:",
    observed_record_count
)


if records_evaluable:

    extracted_df = pd.DataFrame(
        extracted_records
    )

    display(
        extracted_df
    )

In [ ]:
# ============================================================
# 6. Record and content diagnostics
# ============================================================

record_structure_issues = []

field_type_issues = []

missing_mandatory_values = []


# ------------------------------------------------------------
# A. Exact schema
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Record is not a JSON object"
                }
            )

            continue


        observed_fields = list(
            record.keys()
        )


        if observed_fields != EXPECTED_FIELDS:

            record_structure_issues.append(
                {
                    "record_index":
                        record_index,

                    "issue":
                        "Field names or field order differ",

                    "expected_fields":
                        EXPECTED_FIELDS,

                    "observed_fields":
                        observed_fields,

                    "missing_fields":
                        [
                            field
                            for field
                            in EXPECTED_FIELDS
                            if field not in record
                        ],

                    "extra_fields":
                        [
                            field
                            for field
                            in observed_fields
                            if field not in EXPECTED_FIELDS
                        ]
                }
            )


    records_with_structure_issues = len({
        issue["record_index"]
        for issue
        in record_structure_issues
    })


    record_schema_valid = (
        records_with_structure_issues
        == 0
    )


else:

    records_with_structure_issues = None

    record_schema_valid = None


# ------------------------------------------------------------
# B. Field types + mandatory completeness
# ------------------------------------------------------------

if records_evaluable:

    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue


        # -----------------------------------------------
        # String-or-null fields
        # -----------------------------------------------

        for field in STRING_OR_NULL_FIELDS:

            value = record.get(
                field
            )


            if (
                value is not None
                and not isinstance(
                    value,
                    str
                )
            ):

                field_type_issues.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field,

                        "observed_type":
                            type(
                                value
                            ).__name__,

                        "expected_type":
                            "string or null"
                    }
                )


        # -----------------------------------------------
        # Value: number or null
        # -----------------------------------------------

        value = record.get(
            "Value"
        )


        if (
            value is not None
            and (
                isinstance(
                    value,
                    bool
                )
                or not isinstance(
                    value,
                    (int, float)
                )
            )
        ):

            field_type_issues.append(
                {
                    "record_index":
                        record_index,

                    "field":
                        "Value",

                    "observed_type":
                        type(
                            value
                        ).__name__,

                    "expected_type":
                        "number or null"
                }
            )


        # -----------------------------------------------
        # Mandatory content
        # -----------------------------------------------

        for field in MANDATORY_CONTENT_FIELDS:

            field_value = record.get(
                field
            )


            if (
                field_value is None
                or (
                    isinstance(
                        field_value,
                        str
                    )
                    and not field_value.strip()
                )
            ):

                missing_mandatory_values.append(
                    {
                        "record_index":
                            record_index,

                        "field":
                            field
                    }
                )


    records_with_type_issues = len({
        issue["record_index"]
        for issue
        in field_type_issues
    })


    field_types_valid = (
        records_with_type_issues == 0
    )


    missing_mandatory_value_count = len(
        missing_mandatory_values
    )


    mandatory_fields_complete = (
        missing_mandatory_value_count
        == 0
    )


else:

    records_with_type_issues = None

    field_types_valid = None

    missing_mandatory_value_count = None

    mandatory_fields_complete = None


# ------------------------------------------------------------
# C. Count + category
# ------------------------------------------------------------

if records_evaluable:

    record_count_valid = (
        observed_record_count
        == EXPECTED_RECORD_COUNT
    )


    observed_category_counts = dict(
        Counter(
            record.get(
                "Category"
            )
            for record
            in extracted_records
            if isinstance(
                record,
                dict
            )
        )
    )


    categories_valid = set(
        observed_category_counts
    ).issubset(
        ALLOWED_CATEGORIES
    )


    category_counts_valid = (
        observed_category_counts
        == EXPECTED_CATEGORY_COUNTS
    )


else:

    record_count_valid = None

    observed_category_counts = None

    categories_valid = None

    category_counts_valid = None


# ------------------------------------------------------------
# D. Full-record duplicates
# ------------------------------------------------------------

if records_evaluable:

    duplicate_counter = Counter(
        tuple(
            json.dumps(
                record.get(field),
                ensure_ascii=False,
                sort_keys=True
            )
            for field
            in EXPECTED_FIELDS
        )
        for record
        in extracted_records
        if isinstance(
            record,
            dict
        )
    )


    duplicate_records = [
        list(key)
        for key, count
        in duplicate_counter.items()
        if count > 1
    ]


    duplicate_record_count = len(
        duplicate_records
    )


    duplicate_records_absent = (
        duplicate_record_count == 0
    )


else:

    duplicate_records = None

    duplicate_record_count = None

    duplicate_records_absent = None


# ------------------------------------------------------------
# E. Constant semantic fields
# ------------------------------------------------------------

if records_evaluable:

    category_constant_valid = all(
        isinstance(
            record,
            dict
        )
        and record.get(
            "Category"
        )
        == REFERENCE_CATEGORY

        for record
        in extracted_records
    )


    topic_constant_valid = all(
        isinstance(
            record,
            dict
        )
        and record.get(
            "Topic"
        )
        == REFERENCE_TOPIC

        for record
        in extracted_records
    )


    description_constant_valid = all(
        isinstance(
            record,
            dict
        )
        and record.get(
            "Description"
        )
        == REFERENCE_DESCRIPTION

        for record
        in extracted_records
    )


    unit_null_for_all_records = all(
        isinstance(
            record,
            dict
        )
        and record.get(
            "Unit"
        )
        is None

        for record
        in extracted_records
    )


    constant_fields_valid = all([
        category_constant_valid,
        topic_constant_valid,
        description_constant_valid,
        unit_null_for_all_records
    ])


else:

    category_constant_valid = None

    topic_constant_valid = None

    description_constant_valid = None

    unit_null_for_all_records = None

    constant_fields_valid = None


# ------------------------------------------------------------
# F. Reporting-period / target-year diagnostics
# ------------------------------------------------------------

if records_evaluable:

    observed_reporting_years = []


    invalid_reporting_periods = []


    for record_index, record in enumerate(
        extracted_records
    ):

        if not isinstance(
            record,
            dict
        ):
            continue


        period = record.get(
            "Reporting Period"
        )


        try:

            year = int(
                str(
                    period
                ).strip()
            )


            observed_reporting_years.append(
                year
            )


        except (
            ValueError,
            TypeError
        ):

            invalid_reporting_periods.append(
                {
                    "record_index":
                        record_index,

                    "Reporting Period":
                        period
                }
            )


    observed_year_counter = Counter(
        observed_reporting_years
    )


    TARGET_YEAR_SET = set(
        TARGET_YEARS
    )


    OBSERVED_YEAR_SET = set(
        observed_reporting_years
    )


    missing_target_years = sorted(
        TARGET_YEAR_SET
        - OBSERVED_YEAR_SET
    )


    unexpected_extracted_years = sorted(
        OBSERVED_YEAR_SET
        - TARGET_YEAR_SET
    )


    duplicate_extracted_years = sorted(
        year
        for year, count
        in observed_year_counter.items()
        if count > 1
    )


    fixed_years_valid = (
        len(
            missing_target_years
        )
        == 0
        and len(
            duplicate_extracted_years
        )
        == 0
        and len(
            invalid_reporting_periods
        )
        == 0
    )


    unexpected_years_absent = (
        len(
            unexpected_extracted_years
        )
        == 0
    )


else:

    observed_reporting_years = None

    invalid_reporting_periods = None

    missing_target_years = None

    unexpected_extracted_years = None

    duplicate_extracted_years = None

    fixed_years_valid = None

    unexpected_years_absent = None


# ------------------------------------------------------------
# G. Source-location diagnostics
# ------------------------------------------------------------

year_check_rows = []


if records_evaluable:

    source_location_issue_count = 0


    for year in TARGET_YEARS:

        matches = [
            record
            for record
            in extracted_records
            if (
                isinstance(
                    record,
                    dict
                )
                and str(
                    record.get(
                        "Reporting Period"
                    )
                ).strip()
                == str(
                    year
                )
            )
        ]


        expected_location = (
            EXPECTED_SOURCE_LOCATION_BY_YEAR.get(
                year
            )
        )


        if len(matches) == 1:

            record = matches[0]


            observed_location = (
                record.get(
                    "Source Location"
                )
            )


            source_location_valid = (
                observed_location
                == expected_location
            )


            if not source_location_valid:

                source_location_issue_count += 1


            year_check_rows.append(
                {
                    "Year":
                        year,

                    "Record Present":
                        True,

                    "Observed Value":
                        record.get(
                            "Value"
                        ),

                    "Observed Unit":
                        record.get(
                            "Unit"
                        ),

                    "Expected Source Location":
                        expected_location,

                    "Observed Source Location":
                        observed_location,

                    "Source Location Valid":
                        source_location_valid
                }
            )


        else:

            source_location_issue_count += 1


            year_check_rows.append(
                {
                    "Year":
                        year,

                    "Record Present":
                        False,

                    "Observed Value":
                        None,

                    "Observed Unit":
                        None,

                    "Expected Source Location":
                        expected_location,

                    "Observed Source Location":
                        None,

                    "Source Location Valid":
                        False
                }
            )


    source_locations_valid = (
        source_location_issue_count
        == 0
    )


else:

    source_location_issue_count = None

    source_locations_valid = None


year_check_df = pd.DataFrame(
    year_check_rows
)


year_check_df.to_csv(
    YEAR_CHECK_PATH,
    index=False,
    encoding="utf-8-sig"
)


# ------------------------------------------------------------
# H. Numerical-output diagnostics
#
# These are structural/content diagnostics only.
# Exact numerical agreement is reserved for Validation A.
# ------------------------------------------------------------

if records_evaluable:

    numeric_value_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(
                record,
                dict
            )
            and isinstance(
                record.get(
                    "Value"
                ),
                (int, float)
            )
            and not isinstance(
                record.get(
                    "Value"
                ),
                bool
            )
        )
    )


    null_value_count = sum(
        1
        for record
        in extracted_records
        if (
            isinstance(
                record,
                dict
            )
            and record.get(
                "Value"
            )
            is None
        )
    )


else:

    numeric_value_count = None

    null_value_count = None


# ------------------------------------------------------------
# I. D12 content diagnostics
# ------------------------------------------------------------

CONTENT_DIAGNOSTICS = {
    "record_count_matches_reference":
        record_count_valid,

    "categories_valid":
        categories_valid,

    "category_counts_match_reference":
        category_counts_valid,

    "mandatory_fields_complete":
        mandatory_fields_complete,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records_absent":
        duplicate_records_absent,

    "category_constant_valid":
        category_constant_valid,

    "topic_constant_valid":
        topic_constant_valid,

    "description_constant_valid":
        description_constant_valid,

    "unit_null_for_all_records":
        unit_null_for_all_records,

    "constant_fields_valid":
        constant_fields_valid,

    "missing_target_years":
        missing_target_years,

    "unexpected_extracted_years":
        unexpected_extracted_years,

    "duplicate_extracted_years":
        duplicate_extracted_years,

    "invalid_reporting_periods":
        invalid_reporting_periods,

    "fixed_years_valid":
        fixed_years_valid,

    "unexpected_years_absent":
        unexpected_years_absent,

    "source_location_issue_count":
        source_location_issue_count,

    "source_locations_valid":
        source_locations_valid,

    "numeric_value_count":
        numeric_value_count,

    "null_value_count":
        null_value_count
}


print(
    "Record schema valid:",
    record_schema_valid
)

print(
    "Field types valid:",
    field_types_valid
)

print(
    "Observed record count:",
    observed_record_count
)

print(
    "Record count matches:",
    record_count_valid
)

print(
    "Category counts match:",
    category_counts_valid
)

print(
    "Mandatory fields complete:",
    mandatory_fields_complete
)

print(
    "Duplicate records:",
    duplicate_record_count
)

print(
    "Category constant valid:",
    category_constant_valid
)

print(
    "Topic constant valid:",
    topic_constant_valid
)

print(
    "Description constant valid:",
    description_constant_valid
)

print(
    "Unit null for all records:",
    unit_null_for_all_records
)

print(
    "All target years valid:",
    fixed_years_valid
)

print(
    "Unexpected years absent:",
    unexpected_years_absent
)

print(
    "Source locations valid:",
    source_locations_valid
)


print(
    "\nObserved category counts:"
)

print(
    json.dumps(
        observed_category_counts,
        ensure_ascii=False,
        indent=2
    )
    if observed_category_counts
    is not None
    else None
)


display(
    year_check_df
)

In [ ]:
# ============================================================
# 7. Technical diagnostic summary and experiment metadata
# ============================================================

# ------------------------------------------------------------
# Technical/schema validity ONLY
# ------------------------------------------------------------

STRUCTURAL_CHECKS = {
    "valid_json":
        bool(
            valid_json
        ),

    "top_level_object_valid":
        bool(
            top_level_object_valid
        ),

    "document_id_present":
        bool(
            document_id_present
        ),

    "document_id_correct":
        bool(
            document_id_correct
        ),

    "branch_present":
        bool(
            branch_present
        ),

    "branch_correct":
        bool(
            branch_correct
        ),

    "records_present":
        bool(
            records_present
        ),

    "records_is_list":
        bool(
            records_is_list
        ),

    "record_schema_valid":
        (
            record_schema_valid
            if records_evaluable
            else None
        ),

    "field_types_valid":
        (
            field_types_valid
            if records_evaluable
            else None
        )
}

structurally_evaluable = all([
    valid_json,
    top_level_object_valid,
    document_id_present,
    document_id_correct,
    branch_present,
    branch_correct,
    records_present,
    records_is_list,
    record_schema_valid is True,
    field_types_valid is True
])


# ------------------------------------------------------------
# Technical_diagnostics artifact
# ------------------------------------------------------------

TECHNICAL_DIAGNOSTICS = {
    "document_id":
        DOCUMENT_ID,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "input_representation":
        INPUT_REPRESENTATION,

    "valid_json":
        valid_json,

    "json_parsing_error":
        json_parsing_error,

    "top_level_object_valid":
        top_level_object_valid,

    "document_id_present":
        document_id_present,

    "document_id_correct":
        document_id_correct,

    "branch_present":
        branch_present,

    "branch_correct":
        branch_correct,

    "records_present":
        records_present,

    "records_is_list":
        records_is_list,

    "records_evaluable":
        records_evaluable,

    "structural_checks":
        STRUCTURAL_CHECKS,

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_valid":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_valid":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "record_structure_issues":
        (
            record_structure_issues
            if records_evaluable
            else None
        ),

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "field_type_issue_count":
        (
            len(
                field_type_issues
            )
            if records_evaluable
            else None
        ),

    "field_type_issues":
        (
            field_type_issues
            if records_evaluable
            else None
        ),

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "missing_mandatory_values":
        (
            missing_mandatory_values
            if records_evaluable
            else None
        ),

    "duplicate_record_count":
        duplicate_record_count,

    "duplicate_records":
        duplicate_records,

    "content_diagnostics":
        CONTENT_DIAGNOSTICS,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        )
}


TECHNICAL_DIAGNOSTICS_PATH.write_text(
    json.dumps(
        TECHNICAL_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment metadata
# ------------------------------------------------------------

EXPERIMENT_METADATA = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_format":
        SOURCE_FORMAT,

    "source_sha256":
        SOURCE_SHA256,

    "source_structure": {
        "expected_source_row_count":
            EXPECTED_SOURCE_ROW_COUNT,

        "observed_source_row_count":
            OBSERVED_SOURCE_ROW_COUNT,

        "source_row_count_verified":
            SOURCE_ROW_COUNT_VALID,

        "expected_source_columns":
            EXPECTED_SOURCE_COLUMNS,

        "observed_source_columns":
            OBSERVED_SOURCE_COLUMNS,

        "source_columns_verified":
            SOURCE_COLUMNS_VALID,

        "detected_delimiter":
            DETECTED_DELIMITER,

        "expected_min_year":
            EXPECTED_MIN_YEAR,

        "expected_max_year":
            EXPECTED_MAX_YEAR,

        "observed_min_year":
            OBSERVED_MIN_YEAR,

        "observed_max_year":
            OBSERVED_MAX_YEAR,

        "year_sequence_complete":
            YEAR_SEQUENCE_COMPLETE,

        "duplicate_year_count":
            DUPLICATE_YEAR_COUNT,

        "null_cell_count":
            NULL_CELL_COUNT
    },

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "diagnostic_csv_parsing_applied":
        True,

    "diagnostic_delimiter_detection_applied":
        True,

    "diagnostic_numeric_parsing_applied":
        True,

    "diagnostically_parsed_dataframe_used_as_model_input":
        False,

    "derived_representation_used_as_model_input":
        False,

    "csv_to_text_conversion_applied":
        False,

    "row_filtering_applied_to_model_input":
        False,

    "column_filtering_applied_to_model_input":
        False,

    "column_restructuring_applied":
        False,

    "row_reordering_applied":
        False,

    "aggregation_applied":
        False,

    "interpolation_applied":
        False,

    "normalisation_applied":
        False,

    "semantic_rewriting_applied":
        False,

    "unit_inference_applied":
        False,

    "value_conversion_applied":
        False,

    "manual_reconstruction_applied":
        False,

    "manual_correction_applied":
        False,

    "source_content_modification_applied":
        False,

    "complete_original_csv_supplied":
        True,

    "target_years":
        TARGET_YEARS,

    "expected_extraction_scope": {
        "expected_record_count":
            EXPECTED_RECORD_COUNT,

        "expected_category_counts":
            EXPECTED_CATEGORY_COUNTS,

        "expected_fields":
            EXPECTED_FIELDS
    },

    "reference_values_disclosed_to_model":
        False,

    "reference_record_count_explicitly_disclosed_to_model":
        False,

    "target_year_scope_disclosed_to_model":
        True,

    "input_integrity_file":
        INPUT_INTEGRITY_PATH.name,

    "input_integrity_passed":
        INPUT_INTEGRITY_PASSED,

    "representation_file":
        REPRESENTATION_PATH.name,

    "dataset_profile_file":
        DATASET_PROFILE_PATH.name,

    "prompt_file":
        PROMPT_PATH.name,

    "prompt_sha256":
        PROMPT_SHA256,

    "expected_output_format":
        (
            "JSON object with document_id, "
            "branch and records"
        ),

    "execution_environment":
        "Independent ChatGPT conversation",

    "raw_response_file":
        RAW_RESPONSE_PATH.name,

    "raw_response_sha256":
        RAW_RESPONSE_SHA256,

    "parsed_extraction_file":
        (
            PARSED_EXTRACTION_PATH.name
            if parsed_extraction_created
            else None
        ),

    "parsed_extraction_sha256":
        parsed_extraction_sha256,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "observed_record_count":
        observed_record_count,

    "observed_category_counts":
        observed_category_counts,

    "technical_diagnostics_file":
        TECHNICAL_DIAGNOSTICS_PATH.name,

    "year_check_file":
        YEAR_CHECK_PATH.name,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        )

    "notes": (
        "Branch A submits the complete original D12 CSV directly "
        "to the model. Pandas loading, delimiter detection, datatype "
        "inspection and numeric parsing are used only for source "
        "integrity and representation diagnostics and are not "
        "supplied as an alternative model representation. No "
        "CSV-to-text conversion, row filtering, column filtering, "
        "column restructuring, aggregation, interpolation, "
        "normalisation, unit inference, value conversion or manual "
        "correction is applied before extraction. The predefined "
        "target-year list defines the extraction scope and is supplied "
        "to the model, but Stage 1 numerical reference values, "
        "reference source-row mappings and explicit reference record "
        "count are not supplied. Content-level validation is performed separately "
        "in Validation A — D12."
    )
}


EXPERIMENT_METADATA_PATH.write_text(
    json.dumps(
        EXPERIMENT_METADATA,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Experiment summary
# ------------------------------------------------------------

EXPERIMENT_SUMMARY = {
    "document_id":
        DOCUMENT_ID,

    "document_name":
        DOCUMENT_NAME,

    "branch":
        BRANCH,

    "branch_name":
        BRANCH_NAME,

    "source_file":
        SOURCE_PATH.name,

    "source_sha256":
        SOURCE_SHA256,

    "source_verified":
        INPUT_INTEGRITY_PASSED,

    "input_representation":
        INPUT_REPRESENTATION,

    "direct_document_ingestion":
        DIRECT_DOCUMENT_INGESTION,

    "detected_delimiter":
        DETECTED_DELIMITER,

    "source_row_count":
        OBSERVED_SOURCE_ROW_COUNT,

    "source_column_count":
        len(
            OBSERVED_SOURCE_COLUMNS
        ),

    "source_year_range":
        (
            f"{OBSERVED_MIN_YEAR}-"
            f"{OBSERVED_MAX_YEAR}"
        ),

    "source_missing_year_count":
        len(
            MISSING_SOURCE_YEARS
        ),

    "source_duplicate_year_count":
        DUPLICATE_YEAR_COUNT,

    "source_null_cell_count":
        NULL_CELL_COUNT,

    "json_valid":
        valid_json,

    "records_evaluable":
        records_evaluable,

    "structurally_evaluable":
        bool(
            structurally_evaluable
        ),

    "expected_record_count":
        EXPECTED_RECORD_COUNT,

    "observed_record_count":
        observed_record_count,

    "record_count_matches":
        record_count_valid,

    "expected_category_counts":
        EXPECTED_CATEGORY_COUNTS,

    "observed_category_counts":
        observed_category_counts,

    "category_counts_match":
        category_counts_valid,

    "record_schema_valid":
        record_schema_valid,

    "records_with_structure_issues":
        records_with_structure_issues,

    "field_types_valid":
        field_types_valid,

    "records_with_type_issues":
        records_with_type_issues,

    "missing_mandatory_value_count":
        missing_mandatory_value_count,

    "duplicate_record_count":
        duplicate_record_count,

    "category_constant_valid":
        category_constant_valid,

    "topic_constant_valid":
        topic_constant_valid,

    "description_constant_valid":
        description_constant_valid,

    "unit_null_for_all_records":
        unit_null_for_all_records,

    "fixed_years_valid":
        fixed_years_valid,

    "unexpected_years_absent":
        unexpected_years_absent,

    "source_locations_valid":
        source_locations_valid,

    "numeric_value_count":
        numeric_value_count,

    "null_value_count":
        null_value_count,

    "raw_response_preserved":
        RAW_RESPONSE_PATH.exists(),

    "parsed_extraction_created":
        parsed_extraction_created,

    "content_validation_performed":
        False,

    "notes": (
        "This notebook performs source verification, representation "
        "characterisation, D12 Branch A direct-CSV execution "
        "preservation, technical/schema checks and document-specific "
        "scope diagnostics only. Exact agreement of extracted values "
        "with the fixed Stage 1 reference dataset is evaluated "
        "separately in Validation A — D12."
    )
}


EXPERIMENT_SUMMARY_PATH.write_text(
    json.dumps(
        EXPERIMENT_SUMMARY,
        ensure_ascii=False,
        indent=2
    ),
    encoding="utf-8"
)


# ------------------------------------------------------------
# Final status
# ------------------------------------------------------------

print(
    "Structural checks:"
)

print(
    json.dumps(
        STRUCTURAL_CHECKS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\nContent diagnostics:"
)

print(
    json.dumps(
        CONTENT_DIAGNOSTICS,
        ensure_ascii=False,
        indent=2
    )
)


print(
    "\n" + "=" * 60
)

print(
    "D12 Branch A experiment completed"
)

print(
    "=" * 60
)


print(
    "Input integrity passed        :",
    INPUT_INTEGRITY_PASSED
)

print(
    "Detected delimiter            :",
    repr(
        DETECTED_DELIMITER
    )
)

print(
    "Source rows                   :",
    OBSERVED_SOURCE_ROW_COUNT
)

print(
    "Source year range             :",
    OBSERVED_MIN_YEAR,
    "-",
    OBSERVED_MAX_YEAR
)

print(
    "Raw response preserved        :",
    RAW_RESPONSE_PATH.exists()
)

print(
    "Valid JSON                    :",
    valid_json
)

print(
    "Records evaluable             :",
    records_evaluable
)

print(
    "Expected records              :",
    EXPECTED_RECORD_COUNT
)

print(
    "Observed records              :",
    (
        observed_record_count
        if observed_record_count
        is not None
        else "Not evaluable"
    )
)

print(
    "Record count matches          :",
    record_count_valid
)

print(
    "Category counts match         :",
    category_counts_valid
)

print(
    "Record schema valid           :",
    record_schema_valid
)

print(
    "Field types valid             :",
    field_types_valid
)

print(
    "Unit null for all records     :",
    unit_null_for_all_records
)

print(
    "Target years valid            :",
    fixed_years_valid
)

print(
    "Source locations valid        :",
    source_locations_valid
)

print(
    "Structurally evaluable       :",
    structurally_evaluable
)

print(
    "Content validation performed : False"
)

print(
    "Next step                     : Validation A — D12"
)


# ------------------------------------------------------------
# Output existence
#
# Invalid JSON is a legitimate experimental outcome.
# Parsed extraction is therefore conditional.
# ------------------------------------------------------------

required_output_paths = [
    INPUT_INTEGRITY_PATH,
    REPRESENTATION_PATH,
    DATASET_PROFILE_PATH,
    PROMPT_PATH,
    RAW_RESPONSE_PATH,
    TECHNICAL_DIAGNOSTICS_PATH,
    YEAR_CHECK_PATH,
    EXPERIMENT_METADATA_PATH,
    EXPERIMENT_SUMMARY_PATH
]


if (
    parsed_extraction_created
    and PARSED_EXTRACTION_PATH.exists()
):

    required_output_paths.append(
        PARSED_EXTRACTION_PATH
    )


missing_output_files = [
    path.name
    for path
    in required_output_paths
    if not path.exists()
]


if missing_output_files:

    raise AssertionError(
        "Missing output files: "
        f"{missing_output_files}"
    )


print(
    "\nGenerated D12 Branch A files:\n"
)


for path in required_output_paths:

    print(
        "-",
        path.name,
        "| exists:",
        path.exists()
    )